# Experiment: Sionna End-to-End Beamforming Demo

This notebook is a minimal runnable scaffold for a differentiable Sionna + PyTorch beamforming demo. It is intentionally small and only targets a narrowband proof-of-execution path.

Current repository status:
- The notebook structure is ready.
- Local runtime validation is blocked until `sionna` is installed.
- No fake results are reported here.


In [ ]:
from __future__ import annotations

import torch

try:
    import sionna
    SIONNA_AVAILABLE = True
    print("sionna version:", getattr(sionna, "__version__", "unknown"))
except Exception as exc:
    SIONNA_AVAILABLE = False
    print("Sionna is not installed:", exc)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Minimal Plan

1. Create a small learnable beamforming module in PyTorch.
2. Use Sionna-generated or fallback random channels.
3. Optimize negative achievable sum-rate for a few steps.
4. Replace the fallback channel with a real Sionna PHY channel after local installation.


In [ ]:
class TinyBeamformer(torch.nn.Module):
    def __init__(self, num_users: int = 4, num_bs_ant: int = 16):
        super().__init__()
        self.num_users = num_users
        self.num_bs_ant = num_bs_ant
        self.net = torch.nn.Sequential(
            torch.nn.Linear(2 * num_users * num_bs_ant, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2 * num_bs_ant * num_users),
        )

    def forward(self, h):
        x = torch.stack([h.real, h.imag], dim=1).reshape(h.size(0), -1)
        y = self.net(x).reshape(h.size(0), 2, self.num_bs_ant, self.num_users)
        f = torch.complex(y[:, 0].float(), y[:, 1].float())
        power = (torch.abs(f) ** 2).sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
        return f / torch.sqrt(power)

def fallback_channel(batch: int = 64, users: int = 4, antennas: int = 16):
    real = torch.randn(batch, users, antennas, device=device)
    imag = torch.randn(batch, users, antennas, device=device)
    return torch.complex(real, imag) / (2.0 ** 0.5)


In [ ]:
def sum_rate(h, f, noise_var: float = 0.1):
    heff = h @ f
    signal = torch.abs(torch.diagonal(heff, dim1=-2, dim2=-1)) ** 2
    total = torch.abs(heff) ** 2
    interference = total.sum(dim=-1) - signal
    sinr = signal / (interference + noise_var)
    return torch.log2(1 + sinr).sum(dim=-1)

model = TinyBeamformer().to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for step in range(5):
    h = fallback_channel()
    f = model(h)
    rate = sum_rate(h, f).mean()
    loss = -rate
    optim.zero_grad()
    loss.backward()
    optim.step()
    history.append(float(rate.detach().cpu()))
history


## How to Replace the Fallback Channel

After installing Sionna locally, replace `fallback_channel()` with a Sionna PHY narrowband or flat-fading channel generator and keep the same beamformer interface. The repository Python hook is `src/beamforming/data/sionna_generator.py`.
